# Module 4: Why Small Agencies Look Volatile

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

An eight officer department reports zero incidents in June and four in July.
As a percentage that is an infinite increase. As a fact about policing it is
four incidents.

Small agencies swing wildly, and almost all of the swing is arithmetic rather
than behaviour. This notebook shows how much of it is predictable, and builds
the standard tool for comparing agencies of very different sizes without being
fooled by the small ones.

**About 20 minutes.**

## 1. How much do agencies actually swing?

Measure variability as the standard deviation divided by the mean, so that
agencies of different sizes can be compared on the same scale.

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")

clean = monthly[monthly["provisional"] == 0]
clean = clean[~((clean["agency_id"] == "A002") &
                (clean["year_month"] == "2021-06"))]   # the documented unrest month

v = (clean.groupby(["agency_id", "agency_name"])["n_uof"]
     .agg(mean="mean", sd="std").reset_index())
v["variability"] = v["sd"] / v["mean"]
v.sort_values("mean")[["agency_name", "mean", "sd", "variability"]].round(3)

Two Rivers Tribal and Elkhorn swing by more than their own average every month.
Grandview swings by less than a third of its average. The ordering is almost
exactly the ordering by size.

## 2. Most of it is predictable

For counts of events that happen independently, the standard deviation is close
to the square root of the mean. Divide through by the mean and the variability
you should expect from counting alone is one over the square root of the mean.

That is a single line, and it does most of the explaining.

In [ ]:
v["expected_from_counting"] = 1 / np.sqrt(v["mean"])
v["extra"] = v["variability"] - v["expected_from_counting"]

v.sort_values("mean")[["agency_name", "mean", "variability",
                       "expected_from_counting", "extra"]].round(3)

Read the last column. For the two smallest agencies it is close to zero: their
variability is almost entirely counting noise, and there is nothing else in
their monthly numbers to interpret.

For the large agencies it is large. Grandview's variability is about three times
what counting alone would produce, and that excess is real structure, the
seasonal pattern and the trend. A big agency's monthly movements are worth
reading. A small agency's mostly are not.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 4.2))
xs = np.linspace(0.6, 130, 300)
ax.plot(xs, 1 / np.sqrt(xs), color="#eb6834", lw=2, label="counting noise alone")
ax.scatter(v["mean"], v["variability"], s=48, color="#2a78d6", label="each agency")
ax.set_xscale("log")
ax.set_xlabel("average incidents a month")
ax.set_ylabel("standard deviation as a share of the mean")
ax.set_ylim(0, 1.35)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 3. What this does to a ranking

Rank the twelve agencies by their 2023 rate and the small ones scatter to both
ends, not because they are unusual but because they have the least data.

In [ ]:
y23 = (monthly[monthly["year_month"].str[:4] == "2023"]
       .groupby(["agency_id", "agency_name"], as_index=False)
       .agg(uof=("n_uof", "sum"), arrests=("n_arrests", "sum")))
y23["rate"] = 100 * y23["uof"] / y23["arrests"]
y23["agency"] = (y23["agency_name"].str.replace(" Police Department", "", regex=False)
                                   .str.replace(" Sheriff's Office", "", regex=False)
                                   .str.replace(" Police", "", regex=False))

y23.sort_values("rate", ascending=False)[["agency", "arrests", "uof", "rate"]].round(2)

## 4. The funnel plot

The fix is to ask a different question. Instead of "which agency has the highest
rate", ask **"is this agency further from the statewide rate than its own sample
size can explain?"**

An agency with 500 arrests has a wide margin of error. An agency with 49,000 has
a narrow one. The funnel plot draws that margin as a band and puts every agency
inside it.

In [ ]:
state_rate = 100 * y23["uof"].sum() / y23["arrests"].sum()
p = state_rate / 100

# standard error of a proportion, expressed per 100 arrests
y23["se"] = 100 * np.sqrt(p * (1 - p) / y23["arrests"])
y23["low"] = state_rate - 1.96 * y23["se"]
y23["high"] = state_rate + 1.96 * y23["se"]
y23["verdict"] = np.where(y23["rate"] > y23["high"], "above",
                  np.where(y23["rate"] < y23["low"], "below", "within range"))

print(f"statewide rate in 2023: {state_rate:.2f} per 100 arrests\n")
y23.sort_values("arrests")[["agency", "arrests", "rate", "low", "high", "verdict"]].round(2)

This is the result worth pausing on. In the raw ranking Elkhorn sits fourth
and Two Rivers Tribal sixth, above eight larger agencies. The funnel puts both
**within range**: with 484 and 462 arrests you cannot distinguish their rates
from the statewide figure at all.

Meanwhile Grandview sits only ninth in the raw ranking, at 2.32 against a
statewide 2.57, and the funnel flags it as genuinely **below**, because with
49,035 arrests a gap that size can be established. The agencies the funnel picks
out are the large ones, which is the opposite of what a league table does.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
n = np.linspace(300, 60000, 400)
se = 100 * np.sqrt(p * (1 - p) / n)
ax.fill_between(n, state_rate - 1.96 * se, state_rate + 1.96 * se,
                color="#2a78d6", alpha=0.12, label="within range")
ax.axhline(state_rate, color="#8a8880", ls="--", lw=1.3)

inside = y23["verdict"] == "within range"
ax.scatter(y23.loc[inside, "arrests"], y23.loc[inside, "rate"], s=50, color="#2a78d6")
ax.scatter(y23.loc[~inside, "arrests"], y23.loc[~inside, "rate"], s=60, color="#eb6834")
ax.set_xscale("log")
ax.set_xlabel("arrests in 2023")
ax.set_ylabel("use of force per 100 arrests")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 5. More data flags everyone

The funnel is about what the data can establish, not about what matters. Pool
seven years instead of one and the band narrows until almost every agency falls
outside it.

In [ ]:
allyr = (clean[clean["year_month"] <= "2025-12"]
         .groupby(["agency_id", "agency_name"], as_index=False)
         .agg(uof=("n_uof", "sum"), arrests=("n_arrests", "sum")))
allyr["rate"] = 100 * allyr["uof"] / allyr["arrests"]

s7 = 100 * allyr["uof"].sum() / allyr["arrests"].sum()
q = s7 / 100
allyr["se"] = 100 * np.sqrt(q * (1 - q) / allyr["arrests"])
allyr["outside"] = (allyr["rate"] - s7).abs() > 1.96 * allyr["se"]

print(f"pooled over seven years, statewide rate {s7:.2f}")
print(f"agencies outside the band: {int(allyr['outside'].sum())} of {len(allyr)}")

Eleven of twelve. That is not a finding about policing; it is a consequence of
having 765,000 arrests to work with. Once a sample is large enough, every
difference becomes detectable, and the question changes from **can we tell these
apart** to **is the difference big enough to act on**. Always report the size of
the difference next to the verdict.

## 6. A reusable funnel

In [ ]:
def funnel(df, count_col, denom_col, per=100, z=1.96):
    """Flag units whose rate is further from the pooled rate than sampling allows."""
    out = df.copy()
    pooled = per * out[count_col].sum() / out[denom_col].sum()
    prop = pooled / per
    out["pooled_rate"] = pooled
    out["rate"] = per * out[count_col] / out[denom_col]
    out["se"] = per * np.sqrt(prop * (1 - prop) / out[denom_col])
    out["low"] = pooled - z * out["se"]
    out["high"] = pooled + z * out["se"]
    out["verdict"] = np.where(out["rate"] > out["high"], "above",
                      np.where(out["rate"] < out["low"], "below", "within range"))
    out["difference"] = (out["rate"] - pooled).round(2)
    return out

In [ ]:
f = funnel(y23, "uof", "arrests")
f[["agency", "arrests", "rate", "difference", "verdict"]].sort_values("difference").round(2)

## 8. What to carry away

| Habit | Why |
|---|---|
| Show the denominator next to every rate | a rate from 500 arrests is not a rate from 50,000 |
| Compare variability against one over the square root of the mean | it tells you how much of the movement is nothing |
| Use a funnel rather than a league table | it stops small agencies appearing at both extremes |
| Report the size of the difference, not only the verdict | with enough data everything is detectable |
| Never publish a monthly percentage change for a small agency | zero to four is not infinity |

## Exercise

Run the funnel on 2024 and on the two years 2024 and 2025 combined. Does any
agency change verdict, and why?

In [ ]:
# Fill in the blank, then run.
YEARS = None               # try ["2024"] and then ["2024", "2025"]

if YEARS:
    sub = (clean[clean["year_month"].str[:4].isin(YEARS)]
           .groupby(["agency_id", "agency_name"], as_index=False)
           .agg(uof=("n_uof", "sum"), arrests=("n_arrests", "sum")))
    sub["agency"] = sub["agency_name"].str.split().str[0]
    res = funnel(sub, "uof", "arrests")
    print(f"years {YEARS}: pooled rate {res['pooled_rate'].iloc[0]:.2f}")
    print(res[["agency", "arrests", "rate", "difference", "verdict"]]
          .sort_values("arrests").round(2).to_string(index=False))
else:
    print("Set YEARS above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

Agencies move from **within range** to **above** or **below** as the years are
combined, and the ones that move are mid sized. Nothing about them changed
between the two runs: the only difference is that doubling the period roughly
doubles the denominator, which narrows the band by a factor of about the square
root of two.

That is the lesson of the whole module in one experiment. A verdict of "within
range" is a statement about how much data you have, not a clean bill of health,
and a verdict of "above" becomes easier to earn the longer you wait.

</details>

---

**Next:** Part II of the Intermediate series, which takes these clean series and
starts describing them: decomposition, trend measurement, seasonal adjustment
and control limits.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*